In [1]:
# By chat model we mean LLM model which operates with chats 
import os 
import time
import re
import copy
import sys

import json
import operator
from openai import OpenAI

import threading

from langchain_openai import ChatOpenAI 
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

from pydantic import BaseModel, Field

## check performance in agent workflow 
from typing import TypedDict, Annotated, List, Dict, Optional, Set, Literal , Optional, Callable
from langchain_core.messages import BaseMessage, AnyMessage, ToolMessage,HumanMessage, AIMessage, SystemMessage
from langgraph.graph import add_messages , START, END , StateGraph
from IPython.display import Image, display 
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.language_models import BaseChatModel
from langchain_core.tools import tool

from googleapiclient.discovery import build
from urllib.parse import urlparse

# prepare model for embeddings  
from langchain_openai import OpenAIEmbeddings 
from langchain_core.tools import tool, StructuredTool, InjectedToolArg


from sklearn.metrics.pairwise import cosine_similarity
import numpy as np 
import numpy.typing as npt

from tavily import TavilyClient 
from perplexity import Perplexity

import subprocess
from IPython.display import Image, display


d:\MICB_Projects\8_aml_detective\.venv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
## import prompts
from prompts import (
    extract_evidence_claims_prompt , 
    expert_instructions,
    url_summary_instructions,
    system_messages_search_tools,
)


## import Pydantic classes and States 

from state import (
            Journalist, HydePerspectives,
            AllowedClaimType, EvidenceClaim, ClaimsFromSummaries,
            ToolsPool, Tool_Selected,
            LinkCollection, AllowedSeverityLevel, ContentSummary,
            QueryComponentsInState, QueryComponentsCalc,
            DOMAIN_EXCLUDE,
            AllowedEvidenceAssessment,
            AssessEvidenceQuality,
            custom_reducer,
            UnifiedResearchState,
)


In [3]:
## import utilities
from utils import APIVault

## populate classes
key_vault = APIVault()
key_vault.add_key( "openai_key" , os.environ.get("OPENAI_API_KEY")) # personal api key
key_vault.add_key("tavily_key" ,os.environ.get("TAVILY_API_KEY")) # for tavily search and extract
key_vault.add_key("google_key" ,os.environ.get("GOOGLE_API_KEY")) # gor google search engine
key_vault.add_key("openai_key_azure_41" , os.environ.get("OPENAI_API_KEY_AZURE_41")) # for azure project for gpt 4.1
key_vault.add_key("dummy" , os.environ.get("dummy")) # to test error mesage
key_vault.add_key("perplexity_key" , os.environ.get("PERPLEXITY_API_KEY")) # to test error mesage
key_vault.add_key("serp_google_key", os.environ.get("SERP_GOOGLE_API_KEY"))


Key value is empty for 'dummy'


In [4]:
## we need to upload and generate topics

## we also need to generate names variations in different languages
## output become input to geenerate regular expression , google search allows search modifiers but other search tools may not
llm_names_variation = ChatOpenAI(
    model="gpt-4.1-mini",   # note the hyphen
    temperature=0.0,        # pure extraction/translation
    max_tokens=50,          # we only need one short line
    timeout=30,  
    max_retries = 2,
    api_key=key_vault.get_key("openai_key")    
)



qc = QueryComponentsCalc("dummy_name" , DOMAIN_EXCLUDE)
qc.build_all(llm_names_variation)
qstate = qc.to_state()

In [5]:
searc_topics = qstate.search_topics

In [6]:
llm_keywords_generation = ChatOpenAI(
    model="gpt-4.1",     # BETTER: Cheaper, perfectly capable for structured output
    temperature=0.4,         # BEST: Near-deterministic for consistent structure
    max_tokens=10000,         
    timeout=60,             
    max_retries=3,           # ADEQUATE: Structured output usually works first try
    api_key=key_vault.get_key("openai_key")   
)    

In [7]:

class list_words(BaseModel):
    words_string: List[str]

dummy_dict = copy.deepcopy(searc_topics)
n_generate = 100


for topic in searc_topics:
    for lang in searc_topics[topic]: 
        list_topics = ", ".join(searc_topics.get(topic).get(lang))
   
        prompt = f"""You are a keyword expansion specialist for adverse media screening in financial compliance.

CRITICAL: These keywords are used to detect serious criminal activities and compliance violations that pose significant risks to financial institutions.

Topic Category: {topic}
Language: {lang}
Existing Keywords: {list_topics}


Generate {n_generate} additional keywords or phrases that are:
1. Closely related to the existing keywords
2. Specific to "{topic}" domain (avoid generic crime terms)
3. In {lang} language, ro for Romanian, ru for Russian, en for English
4. Maximum 1 words per keyword
5. Suitable for adverse media search queries
6. NOT overlapping with other categories (corruption, organized crime, terrorism, sanctions, etc.)
7. Include variations: plurals, synonyms, related terms (e.g., "fraud" and "fraudulent", "money laundering" and "laundering money")
8. Focus on terms that would appear in serious criminal investigations and adverse media

IMPORTANT: These keywords detect critical compliance issues - be thorough and precise.

Return ONLY the new keywords as a comma-separated list, without explanations or numbering.

New keywords:"""
        
        llm = llm_keywords_generation.with_structured_output(list_words)
        results = llm.invoke(prompt)
        
        # Assign results back to the dictionary
        dummy_dict[topic][lang].extend(results.words_string)
        
        print(f"{topic} ({lang}): Added {len(results.words_string)} keywords")
        print(results.words_string[:10])  # Show first 10 as preview

financial (en): Added 98 keywords
['forgery', 'forgeries', 'bribery', 'briberies', 'counterfeiting', 'counterfeit', 'misappropriation', 'misappropriations', 'skimming', 'skims']
financial (ro): Added 99 keywords
['falsificare', 'falsuri', 'furt', 'deturnare', 'deturnări', 'sustragere', 'sustrageri', 'spălări', 'bani', 'ilegal']
financial (ru): Added 124 keywords
['финансирование', 'обналичивание', 'подлог', 'подделка', 'схема', 'аферист', 'аферисты', 'аферизм', 'махинация', 'махинации']
corruption (en): Added 102 keywords
['graft', 'embezzlement', 'embezzler', 'embezzlers', 'collusion', 'colluder', 'colluders', 'nepotism', 'cronyism', 'favoritism']
corruption (ro): Added 131 keywords
['șpagă', 'mituire', 'corupt', 'corupți', 'corupte', 'corupțiune', 'coruptibil', 'coruptibilitate', 'corupător', 'corupătoare']
corruption (ru): Added 105 keywords
['взяточничество', 'коррупционер', 'коррупционеры', 'коррупционный', 'коррупционность', 'коррупционирование', 'подкуп', 'подкупность', 'подкупн

In [8]:
dummy_dict

{'financial': {'en': ['money laundering',
   'fraud',
   'tax evasion',
   'embezzlement',
   'financial crime',
   'crime',
   'forgery',
   'forgeries',
   'bribery',
   'briberies',
   'counterfeiting',
   'counterfeit',
   'misappropriation',
   'misappropriations',
   'skimming',
   'skims',
   'insider',
   'insiders',
   'front-running',
   'fronts',
   'fronting',
   'fronted',
   'extortion',
   'extortions',
   'diversion',
   'diversions',
   'kickback',
   'kickbacks',
   'ponzi',
   'ponzis',
   'pyramid',
   'pyramids',
   'swindle',
   'swindles',
   'swindling',
   'racketeering',
   'racketeer',
   'racket',
   'rackets',
   'scam',
   'scams',
   'scamming',
   'scammer',
   'scammers',
   'theft',
   'thefts',
   'thefted',
   'pilfering',
   'pilferage',
   'pilferages',
   'pilfered',
   'misuse',
   'misused',
   'misuses',
   'misusing',
   'falsehood',
   'falsehoods',
   'falsification',
   'falsifications',
   'falsifying',
   'falsified',
   'fabrication',
  

In [9]:
## now generate pairs

dummy_dict_bigrams = copy.deepcopy(searc_topics)

for topic in searc_topics:
    for lang in searc_topics[topic]: 
        list_topics = ", ".join(searc_topics.get(topic).get(lang))
        
        prompt = f"""You are a keyword expansion specialist for adverse media screening in financial compliance.

CRITICAL: These keywords are used to detect serious criminal activities and compliance violations that pose significant risks to financial institutions.

Topic Category: {topic}
Language: {lang}
Existing Keywords: {list_topics}

Generate {n_generate} additional keywords or phrases that are:
1. Closely related to the existing keywords
2. Specific to "{topic}" domain (avoid generic crime terms)
3. In {lang} language, ro for Romanian, ru for Russian, en for English
4. You must generate 2 words bigram
5. Suitable for adverse media search queries
6. NOT overlapping with other categories (corruption, organized crime, terrorism, sanctions, etc.)
7. Include variations: plurals, synonyms, related terms (e.g., "fraud" and "fraudulent", "money laundering" and "laundering money")
8. Focus on terms that would appear in serious criminal investigations and adverse media

IMPORTANT: These keywords detect critical compliance issues - be thorough and precise.

Return ONLY the new keywords as a comma-separated list, without explanations or numbering.

New keywords:"""
        
        llm = llm_keywords_generation.with_structured_output(list_words)
        results = llm.invoke(prompt)
        
        # Assign results back to the dictionary
        dummy_dict_bigrams[topic][lang].extend(results.words_string)
        
        print(f"{topic} ({lang}): Added {len(results.words_string)} keywords")
        print(results.words_string[:10])  # Show first 10 as preview

financial (en): Added 99 keywords
['securities fraud', 'accounting fraud', 'investment fraud', 'bank fraud', 'wire fraud', 'credit fraud', 'insurance fraud', 'loan fraud', 'mortgage fraud', 'tax fraud']
financial (ro): Added 98 keywords
['transfer ilegal', 'fonduri nedeclarate', 'conturi offshore', 'fals bancar', 'tranzacții suspecte', 'active ascunse', 'capital ilicit', 'profituri ilegale', 'operațiuni financiare', 'depozite anonime']
financial (ru): Added 98 keywords
['финансовое мошенничество', 'отмывание средств', 'незаконные переводы', 'поддельные счета', 'фиктивные компании', 'финансовые махинации', 'налоговые махинации', 'незаконные выплаты', 'ложные отчеты', 'финансовые злоупотребления']
corruption (en): Added 98 keywords
['graft scheme', 'corrupt official', 'unlawful inducement', 'bribery scheme', 'corrupt payment', 'undue advantage', 'illegal gratuity', 'corrupt practice', 'payoff scheme', 'improper payment']
corruption (ro): Added 98 keywords
['luare mită', 'dare mită', 'fap

In [10]:
dummy_dict_bigrams

{'financial': {'en': ['money laundering',
   'fraud',
   'tax evasion',
   'embezzlement',
   'financial crime',
   'crime',
   'securities fraud',
   'accounting fraud',
   'investment fraud',
   'bank fraud',
   'wire fraud',
   'credit fraud',
   'insurance fraud',
   'loan fraud',
   'mortgage fraud',
   'tax fraud',
   'identity fraud',
   'financial misstatement',
   'asset misappropriation',
   'financial manipulation',
   'market manipulation',
   'insider trading',
   'stock manipulation',
   'securities manipulation',
   'false accounting',
   'financial forgery',
   'check fraud',
   'pyramid scheme',
   'ponzi scheme',
   'forged documents',
   'fake invoices',
   'shell company',
   'front company',
   'ghost account',
   'phantom account',
   'illicit funds',
   'unexplained wealth',
   'suspicious transaction',
   'unusual transaction',
   'structured transaction',
   'layered transaction',
   'false reporting',
   'false statements',
   'falsified records',
   'falsifie

In [11]:
dictionaries_joined = copy.deepcopy(dummy_dict)

# unite result of 2 prompts
for topic in searc_topics:
    for lang in searc_topics[topic]: 
        # Combine keywords from both dictionaries
        dictionaries_joined[topic][lang].extend(dummy_dict_bigrams[topic][lang])
        dictionaries_joined[topic][lang] = list(set(dictionaries_joined[topic][lang]))
        
        print(f"{topic} ({lang}): Total keywords = {len(dictionaries_joined[topic][lang])}")

financial (en): Total keywords = 200
financial (ro): Total keywords = 169
financial (ru): Total keywords = 164
corruption (en): Total keywords = 196
corruption (ro): Total keywords = 180
corruption (ru): Total keywords = 159
organized_crime (en): Total keywords = 202
organized_crime (ro): Total keywords = 202
organized_crime (ru): Total keywords = 202


In [12]:
dictionaries_joined

{'financial': {'en': ['fraudsters',
   'financial manipulation',
   'scams',
   'check fraud',
   'counterfeit cheque',
   'embezzled money',
   'mortgage fraud',
   'front running',
   'thefted',
   'illicit transfer',
   'hidden assets',
   'fronting',
   'forged signature',
   'ghost employee',
   'launderers',
   'unreporting',
   'shell',
   'falsifying',
   'phantom account',
   'fraudulent',
   'fabricated',
   'concealing',
   'fabrication',
   'counterfeiting',
   'illegal lending',
   'insider',
   'skims',
   'pyramids',
   'swindle',
   'falsification',
   'pyramid',
   'ponzi scheme',
   'diversion funds',
   'ghost account',
   'embezzled funds',
   'false reporting',
   'undisclosed',
   'falsehoods',
   'scammer',
   'phantom policy',
   'diverted funds',
   'ponzi',
   'churning account',
   'diversions',
   'pilfering',
   'insurance scam',
   'swindling',
   'evasion',
   'front company',
   'false accounting',
   'scam',
   'embezzlement',
   'financial crime',
   '